# 📊 Dashboard de Vendas Multi-Loja

Análise de desempenho de vendas, faturamento, gerentes e comportamento ao longo do tempo.

## 📥 Importação de bibliotecas

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

## 📂 Carregamento dos dados

In [3]:
#Abrindo os arquivos Excel

vendas_df = pd.read_excel("../data/planilha_vendas/Vendas.xlsx", sheet_name='Plan1')

vendas_dz_df = pd.read_excel("../data/planilha_vendas/Vendas - Dez.xlsx", sheet_name='Plan1')

gerentes_df = pd.read_excel("../data/planilha_vendas/Gerentes.xlsx", sheet_name='Planilha1')

## 🧹 Tratamento dos Dados

- Remoção de valores nulos
- Conversão de datas
- Criação de colunas auxiliares (Ano, Mês, Dia da Semana)
- Junção das tabelas

In [4]:
#juntar as tabelas de vendas
vendas_completo = pd.concat([vendas_df, vendas_dz_df], ignore_index=True)

#juntar a tabela de vendas com a tabela de gerentes
vendas_completo = vendas_completo.merge(gerentes_df, on="ID Loja", how="left")

#Tratar tabela de vendas
vendas_completo = vendas_completo.drop_duplicates()

vendas_completo = vendas_completo.dropna(subset=["Valor Final"])

vendas_completo["Data"] = pd.to_datetime(vendas_completo["Data"])

# Criar colunas de Ano, Mês e Semana
vendas_completo["Ano"] = vendas_completo["Data"].dt.year
vendas_completo["Mes"] = vendas_completo["Data"].dt.month

vendas_completo["Ano_Mes"] = vendas_completo["Data"].dt.to_period("M")

# Mês em português
mapa_meses = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}

vendas_completo["Mes_Nome"] = vendas_completo["Mes"].map(mapa_meses)

# Dia da semana em português
mapa_dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

vendas_completo["Dia_Semana"] = vendas_completo["Data"].dt.day_name().map(mapa_dias)

# Ordenar colunas
colunas_data = ["Data", "Ano", "Mes", "Mes_Nome", "Ano_Mes", "Dia_Semana"]

outras_colunas = [col for col in vendas_completo.columns if col not in colunas_data]

vendas_completo = vendas_completo[colunas_data + outras_colunas]

## 📊 Métricas Principais

Cálculo dos principais indicadores de desempenho:

- Faturamento total
- Faturamento por loja
- Faturamento por gerente
- Produtos mais vendidos
- Ticket médio por loja

In [5]:
faturamento_total = vendas_completo["Valor Final"].sum()

faturamento_loja = vendas_completo.groupby("ID Loja")["Valor Final"].sum().sort_values(ascending=False)

faturamento_gerente = vendas_completo.groupby("Gerente")["Valor Final"].sum().sort_values(ascending=False)

produtos_mais_vendidos = vendas_completo.groupby("Produto")["Quantidade"].sum().sort_values(ascending=False)

ticket_medio_loja = vendas_completo.groupby("ID Loja")["Valor Final"].mean().sort_values(ascending=False)

# Análise por data
analise_dia = vendas_completo.groupby("Data")["Valor Final"].sum().sort_index()

ordem = [
    "Segunda-feira", "Terça-feira", "Quarta-feira",
    "Quinta-feira", "Sexta-feira", "Sábado", "Domingo"
]

analise_dia_semana = (
    vendas_completo.groupby("Dia_Semana")["Valor Final"]
    .sum()
    .reindex(ordem)
)

analise_mes = (
    vendas_completo
    .groupby(["Ano", "Mes", "Mes_Nome"])["Valor Final"]
    .sum()
    .reset_index()  
)

analise_ano = vendas_completo.groupby("Ano")["Valor Final"].sum().sort_index()

## 🏪 Análise por Loja

Avaliação das lojas com maior e menor faturamento.
Identificação de concentração de receita.

In [6]:
valores_formatados_top10 = [
    f"R$ {v:,.0f}".replace(",", ".")
    for v in faturamento_loja.head(10).values
]

valores_formatados_piores = [
    f"R$ {v:,.0f}".replace(",", ".")
    for v in faturamento_loja.tail(5).values
]

#GRÁFICO PRINCIPAL: Top 10 melhores
fig1 = px.bar(
    x=faturamento_loja.head(10).index,
    y=faturamento_loja.head(10).values,
    text=valores_formatados_top10, 
    title="TOP 10 LOJAS COM MAIOR FATURAMENTO (2019)",
    labels={'x': 'Loja', 'y': 'Faturamento (R$)'},
    color=faturamento_loja.head(10).values,
    color_continuous_scale='Greens',
    height=600  
)

fig1.update_traces(
    textposition='outside',
    textfont=dict(size=11, color='white'),
    marker=dict(line=dict(color='white', width=2)),
    customdata=valores_formatados_top10,
    hovertemplate="Loja: %{x}<br>Faturamento: R$ %{customdata}<extra></extra>"
)

fig1.update_yaxes(
    range=[0, faturamento_loja.head(10).max() * 1.15],
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    showgrid=True,
    gridcolor='gray',
    griddash='dot',
    ticklabelposition="outside",
    ticklabelstandoff=12,
    ticks="outside",
    title_standoff=25,
    title_font=dict(size=14, color='white')
)

fig1.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    tickangle=35,  
    tickfont=dict(size=9, color='white'),  
    title_font=dict(size=14, color='white'),
    ticklabelposition="outside"
)

fig1.update_layout(
    margin=dict(t=80, l=120, r=50, b=140),  
    height=600,
    bargap=0.35,
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white', size=12),
    title_font=dict(size=22, color='white'),
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),
    coloraxis_showscale=False
)

fig1.show()

#GRÁFICO COMPLEMENTAR: Top 5 piores
fig_piores = px.bar(
    x=faturamento_loja.tail(5).index,
    y=faturamento_loja.tail(5).values,
    text=valores_formatados_piores,
    title="TOP 5 LOJAS COM MENOR FATURAMENTO (2019)",
    labels={'x': 'Loja', 'y': 'Faturamento (R$)'},
    color=faturamento_loja.tail(5).values,
    color_continuous_scale='Reds',
    height=550  
)

fig_piores.update_traces(
    textposition='outside',
    textfont=dict(size=12, color='white'),  
    customdata=valores_formatados_piores,
    hovertemplate="Loja: %{x}<br>Faturamento: R$ %{customdata}<extra></extra>"
)

fig_piores.update_yaxes(
    range=[0, faturamento_loja.tail(5).max() * 1.25],  
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    showgrid=True,
    gridcolor='gray',
    griddash='dot',
    ticklabelposition="outside",
    ticklabelstandoff=12,
    ticks="outside",
    title_standoff=25,
    title_font=dict(size=14, color='white')  
)

fig_piores.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    tickangle=0,  
    tickfont=dict(size=12, color='white'),  
    title_font=dict(size=14, color='white'),
    title_standoff=35,  
    automargin=True      
)

fig_piores.update_layout(
    margin=dict(t=120, l=120, r=50, b=100), 
    height=550,
    bargap=0.4,  
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white', size=12),  
    title_font=dict(size=22, color='white'),  
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),
    coloraxis_showscale=False
)

fig_piores.show()


## 📈 Evolução do Faturamento Mensal

Este gráfico apresenta a variação do faturamento ao longo dos meses, permitindo identificar tendências de crescimento, queda e possíveis padrões sazonais nas vendas.

A análise temporal é essencial para entender o comportamento do negócio ao longo do tempo e apoiar decisões estratégicas.

In [7]:
analise_mes['Data'] = pd.to_datetime(
    analise_mes['Ano'].astype(str) + '-' +
    analise_mes['Mes'].astype(str) + '-01'
)

analise_mes = analise_mes.sort_values('Data')

mapa_abrev = {
    1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr',
    5: 'Mai', 6: 'Jun', 7: 'Jul', 8: 'Ago',
    9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'
}

meses_abrev = analise_mes['Mes'].map(mapa_abrev)

mapa_completo = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}

meses_completo = analise_mes['Mes'].map(mapa_completo)

valores = analise_mes['Valor Final'].values

valores_formatados = [
    f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    for v in valores
]

#GRÁFICO DE LINHA: Evolução do faturamento mensal
fig2 = px.line(
    x=meses_abrev,
    y=valores,
    title="EVOLUÇÃO DO FATURAMENTO MENSAL (2019)",
    markers=True,
    labels={'x': 'Mês', 'y': 'Faturamento (R$)'}
)

fig2.update_traces(
    line=dict(color='#4CC9F0', width=3),
    marker=dict(
        size=8,
        color='#4361EE',
        line=dict(color='white', width=2)
    ),

    customdata=list(zip(meses_completo, valores_formatados)),

    hovertemplate=
    "Mês: %{customdata[0]}<br>" +
    "Faturamento: R$ %{customdata[1]}" +
    "<extra></extra>"
)

fig2.update_yaxes(
    range=[0, max(valores) * 1.25],

    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',  
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=20,   
    title_standoff=35,

    tickfont=dict(size=12, color='white'),
    title_font=dict(size=14, color='white'),

    automargin=True
)

fig2.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=20,   
    title_standoff=35,

    tickfont=dict(size=12, color='white'),
    title_font=dict(size=14, color='white'),

    automargin=True
)

fig2.update_layout(
    margin=dict(t=120, l=130, r=50, b=100),
    height=550,

    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white'),

    title_font_size=20,
    title_x=0.5,

    hovermode='closest',  

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    )
)

fig2.show()

## 👔 Análise por Gerente

Comparação de desempenho entre gerentes e identificação dos melhores resultados.

In [8]:
valores_formatados_gerente = [
    f"R$ {v:,.0f}".replace(",", ".")  
    for v in faturamento_gerente.values
]

#Grafico de barras horizontal para faturamento por gerente
fig_gerente = px.bar(
    x=faturamento_gerente.values,
    y=faturamento_gerente.index,
    text=valores_formatados_gerente,
    orientation='h',
    title="FATURAMENTO POR GERENTE (2019)",
    labels={'y': 'Gerente', 'x': 'Faturamento (R$)'},
    color=faturamento_gerente.values,

    color_continuous_scale=[
        "#081C15",
        "#1B4332",
        "#2D6A4F",
        "#40916C",
        "#52B788"
    ],

    height=800
)

fig_gerente.update_traces(
    textposition='outside',  
    textfont=dict(size=11, color='white'),

    marker=dict(
        line=dict(color='white', width=2)
    ),

    cliponaxis=False,
    insidetextanchor='end',

    customdata=valores_formatados_gerente,
    hovertemplate="Gerente: %{y}<br>Faturamento: %{customdata}<extra></extra>"
)

fig_gerente.update_xaxes(
    range=[0, faturamento_gerente.max() * 1.15],
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=12,
    title_standoff=25
)

fig_gerente.update_yaxes(
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    tickfont=dict(size=11),

    ticks="outside",
    ticklen=8,
    tickwidth=2,
    tickcolor="white",

    ticklabelstandoff=8,
    automargin=True
)

fig_gerente.update_layout(
    margin=dict(t=80, l=150, r=50, b=80),
    bargap=0.3,

    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white'),

    title_font_size=22,
    title_x=0.5,

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),

    coloraxis_showscale=False
)

fig_gerente.show()

## 📦 Produtos Mais Vendidos

Análise dos produtos com maior volume de vendas.

In [9]:
#Top 10 produtos mais vendidos
top_produtos = produtos_mais_vendidos.head(10)

valores_formatados_produtos = [
    f"{v:,.0f}".replace(",", ".")
    for v in top_produtos.values
]

fig_produtos = px.bar(
    x=top_produtos.index,
    y=top_produtos.values,
    text=valores_formatados_produtos,
    title="TOP 10 PRODUTOS MAIS VENDIDOS (2019)",
    labels={'x': 'Produto', 'y': 'Quantidade Vendida'},
    color=top_produtos.values,

    color_continuous_scale=[
        "#081C15",
        "#1B4332",
        "#2D6A4F",
        "#40916C",
        "#52B788"
    ],

    height=600
)

fig_produtos.update_traces(
    textposition='outside',
    textfont=dict(size=11, color='white'),

    marker=dict(
        line=dict(color='white', width=2)
    ),

    customdata=valores_formatados_produtos,
    hovertemplate=
    "Produto: %{x}<br>" +
    "Quantidade: %{customdata}" +
    "<extra></extra>"
)

fig_produtos.update_yaxes(
    range=[0, top_produtos.max() * 1.3],
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=12,
    title_standoff=25
)

fig_produtos.update_xaxes(
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    tickangle=35,  
    tickfont=dict(size=10),

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=10,
    title_standoff=30,
    automargin=True
)

fig_produtos.update_layout(
    margin=dict(t=100, l=110, r=50, b=150),  
    bargap=0.35,

    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white'),

    title_font_size=22,
    title_x=0.5,

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),

    coloraxis_showscale=False
)

fig_produtos.show()

## 📅 Faturamento por Dia da Semana

Avaliação de quais dias apresentam maior volume de vendas.

In [10]:
valores_formatados_semana = [
    f"R$ {v:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".") 
    for v in analise_dia_semana.values
]

#Gráfico de barras para faturamento por dia da semana
fig_semana = px.bar(
    x=analise_dia_semana.index,
    y=analise_dia_semana.values,
    text=valores_formatados_semana,
    title="FATURAMENTO POR DIA DA SEMANA",
    labels={'x': 'Dia da Semana', 'y': 'Faturamento (R$)'},
    color=analise_dia_semana.values,
    color_continuous_scale='Blues',
    height=500
)

fig_semana.update_traces(
    textposition='outside',
    textfont=dict(size=11, color='white'),
    marker=dict(line=dict(color='white', width=2)),
    cliponaxis=False,
    customdata=valores_formatados_semana,
    hovertemplate="Dia: %{x}<br>Faturamento: %{customdata}<extra></extra>"
)

fig_semana.update_xaxes(
    tickangle=0,
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=12,
    title_standoff=25
)

fig_semana.update_yaxes(
    range=[0, max(analise_dia_semana.values) * 1.25],
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=15,
    title_standoff=30,

    automargin=True
)

fig_semana.update_layout(
    margin=dict(l=120, r=40, t=80, b=90),
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white'),
    title_x=0.5,
    coloraxis_showscale=False,

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    )
)

fig_semana.show()

## 📆 Faturamento por Ano

Comparação do desempenho anual.

In [11]:
valores_formatados_ano = [
    f"R$ {v:,.0f}".replace(",", ".")
    for v in analise_ano.values
]

#Gráfico de barras para faturamento por ano
fig_ano = px.bar(
    x=analise_ano.index.astype(str),
    y=analise_ano.values,
    text=valores_formatados_ano,
    title="FATURAMENTO POR ANO",
    labels={'x': 'Ano', 'y': 'Faturamento (R$)'},
    color=analise_ano.values,
    color_continuous_scale='Purples',
    height=500
)

fig_ano.update_traces(
    textposition='outside',
    textfont=dict(size=12, color='white'),
    marker=dict(line=dict(color='white', width=2)),
    cliponaxis=False,
    hovertemplate="Ano: %{x}<br>Faturamento: %{customdata}<extra></extra>",
    customdata=valores_formatados_ano
)

fig_ano.update_yaxes(
    range=[0, analise_ano.max() * 1.3],  
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=15,
    title_standoff=30,

    automargin=True
)

fig_ano.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=12,
    title_standoff=25
)

fig_ano.update_layout(
    margin=dict(l=120, r=40, t=80, b=80),
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white'),
    title_x=0.5,
    coloraxis_showscale=False,

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    )
)

fig_ano.show()

## 💰 Ticket Médio por Loja

Análise do valor médio gasto por cliente em cada loja.

In [12]:
top_ticket = ticket_medio_loja.head(10)

valores_formatados_ticket = [
    f"R$ {v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    for v in top_ticket.values
]

#Gráfico de barras para ticket médio por loja
fig_ticket = px.bar(
    x=top_ticket.index,
    y=top_ticket.values,
    text=valores_formatados_ticket,
    title="TOP 10 LOJAS - MAIOR TICKET MÉDIO",
    labels={'x': 'Loja', 'y': 'Ticket Médio (R$)'},
    color=top_ticket.values,
    color_continuous_scale='Teal',
    height=600
)

fig_ticket.update_traces(
    textposition='outside',
    textfont=dict(size=11, color='white'),
    marker=dict(line=dict(color='white', width=2)),
    cliponaxis=False,
    customdata=valores_formatados_ticket,
    hovertemplate="Loja: %{x}<br>Ticket Médio: %{customdata}<extra></extra>"
)

fig_ticket.update_xaxes(
    tickangle=35,
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=8,   
    title_standoff=15,     

    automargin=True
)

fig_ticket.update_yaxes(
    color='white',

    showline=True,
    linecolor='white',
    linewidth=2,

    showgrid=True,
    gridcolor='rgba(200,200,200,0.3)',
    griddash='dot',

    ticks="outside",
    ticklen=6,
    tickwidth=2,

    ticklabelstandoff=15,
    title_standoff=30,

    automargin=True,

    range=[0, top_ticket.values.max() * 1.25]
)

fig_ticket.update_layout(
    margin=dict(t=80, l=120, r=50, b=180),  

    bargap=0.35,

    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white', size=12),

    title_font=dict(size=22, color='white'),
    title_x=0.5,

    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),

    coloraxis_showscale=False
)

fig_ticket.show()